# Q8. Single-sourced vs multi-sourced materials

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** What share of Net PO value is spent on single-sourced materials (only one supplier)
versus multi-sourced materials?

We answer it at two levels:
1. **SAP material number**, the literal answer from the data.
2. **Product family** (see Q4), which is closer to the business meaning, because students created their
   own material numbers for the same product.

Items with no supplier are counted as "no supplier recorded".

In [ ]:
w["family"] = product_family(w)

def sourcing(key):
    n = w.groupby(key, dropna=False)["Supplier"].transform("nunique")
    return np.select([n == 0, n == 1], ["No supplier recorded", "Single-sourced"], "Multi-sourced")

res = {}
for label, key in [("SAP material number", "Material"), ("Product family", "family")]:
    w["sourcing"] = sourcing(key)
    t = w.groupby("sourcing").agg(net=("Net Order Value", "sum"), materials=(key, "nunique"),
                                  items=("Item", "size"))
    t["value_share"] = t["net"] / t["net"].sum()
    res[label] = t
    print(label); display(t.style.format({"net": "{:,.0f}", "value_share": "{:.1%}"}))

In [ ]:
order = ["Single-sourced", "Multi-sourced", "No supplier recorded"]
colors = [SERIES[1], SERIES[0], MUTED]
fig, ax = plt.subplots(figsize=(10, 2.6))
levels = list(res)[::-1]
for j, lvl in enumerate(levels):
    left = 0
    for name, color in zip(order, colors):
        v = res[lvl]["value_share"].get(name, 0)
        ax.barh(lvl, v, left=left, color=color, height=0.55, edgecolor="#fcfcfb", lw=2,
                label=name if j == 0 else None)
        if v > 0.05:
            ax.text(left + v / 2, j, f"{v:.0%}", ha="center", va="center", color="white", fontweight="bold")
        left += v
ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 1); ax.grid(visible=False)
ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, -0.18))
ax.set_title(f"Share of Net PO value by sourcing status, {PERIOD}")
save(fig, "q8_single_vs_multi")

In [ ]:
fam = (w.groupby("family").agg(net=("Net Order Value", "sum"), suppliers=("Supplier", "nunique"),
                               material_numbers=("Material", "nunique"))
         .assign(share=lambda d: d.net / d.net.sum()).sort_values("net", ascending=False))
fam.style.format({"net": "{:,.0f}", "share": "{:.1%}"})

In [ ]:
w["sourcing"] = sourcing("Material")
single = (w[w["sourcing"] == "Single-sourced"]
          .groupby(["Material", "Short Text", "supplier"])["Net Order Value"].sum()
          .sort_values(ascending=False).head(10).reset_index())
single["share_of_total"] = single["Net Order Value"] / TOTAL
single.style.format({"Net Order Value": "{:,.0f}", "share_of_total": "{:.1%}"}).hide(axis="index")

## Findings for Antonio

**Finding.** The answer depends on how "material" is defined:
- By **SAP material number**, **62.7% of value** ($1.92M) is single-sourced: 427 of 489 material numbers have only one supplier.
- By **product family**, only **2.0%** is single-sourced. That is the water bottle cage: one supplier, $62k. The other 98% is multi-sourced (e.g. chain locks from 172 suppliers).

The gap exists because almost every supplier got its own material number for the same product (183 numbers for chain locks).

**Decision it supports.** The real single-source risk is narrow. It covers the water bottle cage and, effectively, finished bikes, where one supplier has 85% (Q4). Those items need a second source. The bigger issue is master data: harmonising material numbers would make sourcing and price comparison possible.

**Limitation / extra data.** Multi-sourced on paper doesn't mean the suppliers are equivalent. Antonio would need quality approval or qualification status per supplier and material, plus info records or contracts, to know which alternative sources are really usable.